# Phase 4 & 5: Data Preprocessing & Feature Engineering
## Scikit-Learn Pipeline & ColumnTransformer Architecture

This notebook demonstrates the end-to-end preprocessing pipeline:
1. Feature extraction from timestamps (Hour, Day of Week, Rush Hour, Night/Day, Season)
2. Weather aggregation & physical hazard flags (Freezing, Low Visibility, High Wind)
3. Purging leakage columns and uninformative metadata
4. Stratified Train / Validation / Test split (70 / 15 / 15) executed strictly before scaling or resampling
5. Building a production `ColumnTransformer` with median imputation and one-hot encoding


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src.data_loader import load_dataset
from src.preprocessor import prepare_raw_data, get_feature_lists, build_preprocessor_pipeline, stratified_split

# Load raw data
raw_df = load_dataset(sample_size=30000)
print("Raw input shape:", raw_df.shape)


### 1. Apply Feature Engineering and Leakage Purging

In [ ]:
clean_df = prepare_raw_data(raw_df)
print("Cleaned DataFrame shape:", clean_df.shape)
clean_df.head(3)


### 2. Stratified Data Split (70% Train, 15% Val, 15% Test)

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(clean_df)
print(f"Train samples: {len(X_train)} | Serious rate: {y_train.mean():.3f}")
print(f"Val samples:   {len(X_val)} | Serious rate: {y_val.mean():.3f}")
print(f"Test samples:  {len(X_test)} | Serious rate: {y_test.mean():.3f}")


### 3. Build & Fit ColumnTransformer on Training Data

In [ ]:
num_cols, cat_cols, road_cols = get_feature_lists(X_train)
preprocessor = build_preprocessor_pipeline(num_cols, cat_cols, road_cols, scale_numeric=True)

# Fit strictly on train
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

print("Processed feature matrix shape:", X_train_proc.shape)
